## **Weather Problem**
# **Regression problems using multiple linear regression, predicted apparent temperature for given temperature, humidity, wind speed, visibility, pressure**


# **Loading the dataset**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error,mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

# **Uploading the data from local to colab**


In [2]:
from google.colab import files
uploaded=files.upload()

Saving Weather_Data.csv to Weather_Data.csv


## **Reading CSV file data**

In [3]:
data=pd.read_csv('Weather_Data.csv')


## **Data Head**

In [4]:
data.head()

,Date/Time,Temp_C,Dew Point Temp_C,Rel Hum_%,Wind Speed_km/h,Visibility_km,Press_kPa,Weather
0,1/1/2012 0:00,-1.8,-3.9,86,4,8.0,101.24,Fog
1,1/1/2012 1:00,-1.8,-3.7,87,4,8.0,101.24,Fog
2,1/1/2012 2:00,-1.8,-3.4,89,7,4.0,101.26,"Freezing Drizzle,Fog"
3,1/1/2012 3:00,-1.5,-3.2,88,6,4.0,101.27,"Freezing Drizzle,Fog"
4,1/1/2012 4:00,-1.5,-3.3,88,7,4.8,101.23,Fog


## **Data Information**

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Date/Time         8784 non-null   object 
 1   Temp_C            8784 non-null   float64
 2   Dew Point Temp_C  8784 non-null   float64
 3   Rel Hum_%         8784 non-null   int64  
 4   Wind Speed_km/h   8784 non-null   int64  
 5   Visibility_km     8784 non-null   float64
 6   Press_kPa         8784 non-null   float64
 7   Weather           8784 non-null   object 
dtypes: float64(4), int64(2), object(2)
memory usage: 549.1+ KB


# **Pre-Processing Code**

In [ ]:
# # Rename columns
# df.columns = ['Datetime', 'Temp [C]', 'Dew Point Temp [C]', 'Rel Hum [%]', 'Wind Speed [km/h]', 'Visibility [km]', 'Press [kPa]', 'Weather']

# # Convert time stamp column to time format
# df.index = pd.to_datetime(df['Datetime']).dt.floor('T')
# df = df.iloc[:, 1:]

# # Remove empty rows (if any are present in the data).
# df.drop(df[df.isnull().any(axis = 1)].index, inplace = True)

# # Remove duplicates (if any are present in the dataset).
# df.drop_duplicates(inplace = True)

# # Separate 'Weather' column into three separate parts (each description of weather conditions in a separate column)
# weather_split = ['Weather - p. 1', 'Weather - p. 2', 'Weather - p. 3']
# df[weather_split] = df['Weather'].str.split(',', expand = True)

# # Remove 'Weather' column (redundant).
# df.drop(['Weather'], axis = 1, inplace = True)

# # Storage of data on weather conditions according to zero-one coding.
# weather_category_list = np.array([])

# for column in df[weather_split]:
#     weather_category_list = np.append(weather_category_list, df[weather_split][column].unique())

# weather_category_list = weather_category_list[weather_category_list != None]
# weather_category_list = np.unique(weather_category_list)

# df[weather_category_list] = 0

# for column in df[weather_split]:
#     for index in df[weather_split].index:
#         if df.loc[index, column] != None:
#             df.at[index, df.loc[index, column]] = df.loc[index, df.loc[index, column]] + 1

# # Delete 'Weather - p. 1', 'Weather - p. 2' and 'Weather - p. 3' columns (redundant)
# df.drop(weather_split, axis = 1, inplace = True)

# # Create an auxiliary set of column names in measured values
# weather_measurement_data = ['Temp [C]', 'Dew Point Temp [C]', 'Rel Hum [%]', 'Wind Speed [km/h]', 'Visibility [km]', 'Press [kPa]']

## **Checking for null values in dataset**

In [8]:
data.isnull().sum()

,0
Date/Time,0
Temp_C,0
Dew Point Temp_C,0
Rel Hum_%,0
Wind Speed_km/h,0
Visibility_km,0
Press_kPa,0
Weather,0


## **Checking for redundant  data**

In [9]:
data.duplicated()

,0
0,False
1,False
2,False
3,False
4,False
...,...
8779,False
8780,False
8781,False
8782,False


## **Data Pre-processing**

In [10]:
# Converting 'Date/Time' to datetime format and extracting useful information
data['Date/Time']=pd.to_datetime(data['Date/Time'])

data['Hour']=data['Date/Time'].dt.hour
data['Day']=data['Date/Time'].dt.day
data['Month']=data['Date/Time'].dt.month
data['weekday']=data['Date/Time'].dt.weekday

#now that we extracted  useful information from Date/Time column we can drop that column
X=data.drop(columns=['Date/Time'])

#converting weather column from categorical to numberical values using dummies
X = pd.get_dummies(X, columns=['Weather'], drop_first=True)


## **Dependent and Independent variable**


In [13]:
# Define the target variable (e.g., predicting 'Temp_C')
X_simple=data[['Dew Point Temp_C']]
y = data['Temp_C']

## **Spliting Dataset**

In [14]:
#Split Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X_simple, y, test_size=0.2, random_state=42)

## **Scaling numerical features**

In [15]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## **Ridge Regression Model**

In [19]:
ridge = Ridge()

# Hyperparameter Grid (alpha for Ridge)
param_grid = {'alpha': [0.01, 0.1, 1, 10, 100]}

# Perform GridSearchCV to tune the hyperparameters
grid_search = GridSearchCV(estimator=ridge, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search.fit(X_train_scaled, y_train)


GridSearchCV(cv=5, estimator=Ridge(),
             param_grid={'alpha': [0.01, 0.1, 1, 10, 100]},
             scoring='neg_mean_squared_error')

## **Best Hyperparameters**

In [20]:
best_alpha = grid_search.best_params_['alpha']
best_score = grid_search.best_score_

## **Train the best model using the best alpha**

In [21]:
best_model = grid_search.best_estimator_
best_model.fit(X_train_scaled, y_train)

Ridge(alpha=0.1)

## **Predicting temperature**

In [22]:
y_pred=best_model.predict(X_test_scaled)

In [23]:
for i ,prediction in enumerate(y_pred):
  print(f"weather sample {i+1} predicted accumated temperature {prediction}")

weather sample 1 predicted accumated temperature 15.821422412730042
weather sample 2 predicted accumated temperature 3.001971064662449
weather sample 3 predicted accumated temperature 16.0217263400436
weather sample 4 predicted accumated temperature 7.809265320187797
weather sample 5 predicted accumated temperature -12.62173526579493
weather sample 6 predicted accumated temperature 23.533123614301957
weather sample 7 predicted accumated temperature -6.41231351907469
weather sample 8 predicted accumated temperature 16.722790085641044
weather sample 9 predicted accumated temperature 16.822942049297822
weather sample 10 predicted accumated temperature -14.424470611616934
weather sample 11 predicted accumated temperature 14.218990994221594
weather sample 12 predicted accumated temperature 21.63023630482317
weather sample 13 predicted accumated temperature 24.634795214526513
weather sample 14 predicted accumated temperature 1.4996916098107773
weather sample 15 predicted accumated temperatur

## **Evaluate Model Performance**

In [24]:
# Calculate performance metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse}")
print(f"R² Score: {r2}")


Mean Squared Error (MSE): 18.631761787273646
R² Score: 0.8713486548457821


## **Multiple linear regression**

In [25]:
# Select multiple independent variables
y = data["Temp_C"]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)




## **Scaling the features**

In [26]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## **Ridge Regression Model**

In [27]:
ridge = Ridge()

# Hyperparameter Grid (alpha for Ridge)
param_grid = {'alpha': [0.01, 0.1, 1, 10, 100]}

# Perform GridSearchCV to tune the hyperparameters
grid_search = GridSearchCV(estimator=ridge, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search.fit(X_train_scaled, y_train)


GridSearchCV(cv=5, estimator=Ridge(),
             param_grid={'alpha': [0.01, 0.1, 1, 10, 100]},
             scoring='neg_mean_squared_error')

## **Best Hyperparameters**

In [28]:
best_alpha = grid_search.best_params_['alpha']
best_score = grid_search.best_score_

## **Train the best model using the best alpha**

In [29]:
best_model = grid_search.best_estimator_
best_model.fit(X_train_scaled, y_train)

Ridge(alpha=0.01)

## **Predicting the temp**

In [30]:
# Make predictions
y_pred=best_model.predict(X_test_scaled)


In [32]:
for i ,prediction in enumerate(y_pred):
  print(f"weather sample {i+1} predicted accumated temperature {prediction}")

weather sample 1 predicted accumated temperature 12.799941743409008
weather sample 2 predicted accumated temperature 1.000037217071208
weather sample 3 predicted accumated temperature 12.599917544411541
weather sample 4 predicted accumated temperature 11.700181800597758
weather sample 5 predicted accumated temperature -11.599677541383455
weather sample 6 predicted accumated temperature 21.100000674265097
weather sample 7 predicted accumated temperature -9.600028838547052
weather sample 8 predicted accumated temperature 19.600155744953337
weather sample 9 predicted accumated temperature 24.19972144098726
weather sample 10 predicted accumated temperature -15.199849977185513
weather sample 11 predicted accumated temperature 17.60021584566504
weather sample 12 predicted accumated temperature 22.60022378246648
weather sample 13 predicted accumated temperature 20.999940413954878
weather sample 14 predicted accumated temperature 3.0001805508369754
weather sample 15 predicted accumated tempera

## **Evaluate Model Performance**


In [33]:
# Calculate performance metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse}")
print(f"R² Score: {r2}")


Mean Squared Error (MSE): 7.128190685546784e-08
R² Score: 0.9999999995078022


## **Apprant temperature (Feels like temperatures)**

# **Since There is no apparent temprature column is your data set first we will compute apprant temprature column scientific formula to calculate apprant temprature**
# The apparent temperature (feels-like temperature) is calculated using meteorological models that take into account:
# New section
# Temperature
# Humidity
# Wind Speed

# The formula used by the Meteorology is:https://www.vcalc.com/wiki/australian-apparent-temperature



In [34]:

# Convert Wind Speed from km/h to m/s
data['Wind Speed_m/s']=data['Wind Speed_km/h']/3.6

# Calculate water vapor pressure (e)
data["e"] = (data["Rel Hum_%"] / 100) * (6.105 * np.exp((17.27 * data["Temp_C"]) / (data["Temp_C"] + 237.7)))


# Calculate Apparent Temperature
data["Apparent_Temp_C"] = data["Temp_C"] + (0.33 * data["e"]) - (0.70 * data["Wind Speed_m/s"]) - 4.00


# Display first few rows
print(data[["Temp_C", "Rel Hum_%", "Wind Speed_km/h", "Apparent_Temp_C"]].head())



   Temp_C  Rel Hum_%  Wind Speed_km/h  Apparent_Temp_C
0    -1.8         86                4        -5.059090
1    -1.8         87                4        -5.041431
2    -1.8         89                7        -5.589446
3    -1.5         88                6        -5.077932
4    -1.5         88                7        -5.272376


## **Selecting feature**

In [36]:
X=data[['Temp_C', 'Rel Hum_%', 'Wind Speed_km/h', 'Visibility_km', 'Press_kPa']]
y=data['Apparent_Temp_C']


## **Splititng data**

In [37]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## **Scaling Features**

In [38]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## **Ridge Regression Model**

In [39]:
ridge = Ridge()

# Hyperparameter Grid (alpha for Ridge)
param_grid = {'alpha': [0.01, 0.1, 1, 10, 100]}

# Perform GridSearchCV to tune the hyperparameters
grid_search = GridSearchCV(estimator=ridge, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search.fit(X_train_scaled, y_train)

GridSearchCV(cv=5, estimator=Ridge(),
             param_grid={'alpha': [0.01, 0.1, 1, 10, 100]},
             scoring='neg_mean_squared_error')

## **Best Hyperparameter**

In [40]:
best_alpha = grid_search.best_params_['alpha']
best_score = grid_search.best_score_

## **Train the best model using the best alpha**

In [41]:
best_model = grid_search.best_estimator_
best_model.fit(X_train_scaled, y_train)

Ridge(alpha=0.01)

## **Predicting the Apprent Temp**

In [42]:
# Make predictions
y_pred=best_model.predict(X_test_scaled)

In [43]:
for i ,prediction in enumerate(y_pred):
  print(f"Weather sample {i+1} apprent temprature(feels like temp) :{prediction} ")

Weather sample 1 apprent temprature(feels like temp) :9.70602793543559 
Weather sample 2 apprent temprature(feels like temp) :-5.387564503803282 
Weather sample 3 apprent temprature(feels like temp) :8.232488662010832 
Weather sample 4 apprent temprature(feels like temp) :6.778728582743797 
Weather sample 5 apprent temprature(feels like temp) :-17.798039167272982 
Weather sample 6 apprent temprature(feels like temp) :20.508739034581144 
Weather sample 7 apprent temprature(feels like temp) :-14.354861782301445 
Weather sample 8 apprent temprature(feels like temp) :15.225064559140062 
Weather sample 9 apprent temprature(feels like temp) :20.599941734807143 
Weather sample 10 apprent temprature(feels like temp) :-25.058986033651458 
Weather sample 11 apprent temprature(feels like temp) :15.414155927620314 
Weather sample 12 apprent temprature(feels like temp) :20.07069460656225 
Weather sample 13 apprent temprature(feels like temp) :20.631272739886334 
Weather sample 14 apprent temprature

## **Evaluating the model**

In [44]:

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"R² Score: {r2}")

MAE: 0.5478452089663562
MSE: 0.48429794391075914
R² Score: 0.9975812393380608
